This notebook explores the effect of wind on the antennae of the fly. Here, each antenna is connected to the head with a ball joint. The wind applies a force on the antenna (for details, see [the MuJoCo documentation on inertia-based model offluid dynamics](https://mujoco.readthedocs.io/en/stable/computation/fluid.html#inertia-model)), which causes it to deflect. When the wind stops, the antenna will return to its original position due to the stiffness of the joint. We will simulate this process and visualize the antenna deflection over time.

In [1]:
import cv2
from miniproject.simulation import create_fly
from flygym.compose import ActuatorType
from flygym.compose.world import FlatGroundWorld
from flygym.utils.math import Rotation3D
from flygym import Simulation
import numpy as np

In [2]:
fly = create_fly()

cam_kwargs_front = {
    "name": "front_cam",
    "mode": "fixed",
    "pos_offset": (2.5, 0, 1.25),
    "rotation": Rotation3D("euler", (np.pi / 2, 0, np.pi / 2)),
    "fovy": 40,
}

cam_kwargs_top = {
    "name": "top_cam",
    "mode": "fixed",
    "pos_offset": (1, 0, 2.5),
    "rotation": Rotation3D("euler", (0, 0, -np.pi / 2)),
    "fovy": 40,
}

cam_kwargs_right = {
    "name": "right_cam",
    "mode": "fixed",
    "pos_offset": (1, -1, 1.25),
    "rotation": Rotation3D("euler", (np.pi / 2, 0, 0)),
    "fovy": 40,
}
cam_front = fly.add_tracking_camera(**cam_kwargs_front)
cam_top = fly.add_tracking_camera(**cam_kwargs_top)
cam_right = fly.add_tracking_camera(**cam_kwargs_right)

In [3]:
world = FlatGroundWorld()
checker = world.mjcf_root.find("texture", "checker")
checker.rgb1 = (1, 1, 1)
checker.rgb2 = (1, 1, 1)
world.add_fly(
    fly, spawn_position=(0, 0, 0.1), spawn_rotation=Rotation3D("quat", (1, 0, 0, 0))
)

In [ ]:
wind_magnitude = 60000

sim = Simulation(world)
sim.set_renderer([cam_front, cam_top, cam_right], camera_res=(256, 256))
sim.reset()

for i in range(2000):
    sim.step()

sim.set_actuator_inputs(fly.name, ActuatorType.ADHESION, np.array([1.0] * 6))

for i in range(8):
    angle_deg = 45 * i
    sim.set_wind(wind_magnitude, angle_deg)

    for _ in range(2000):
        sim.step()
        if sim.render_as_needed():
            for frames in sim.renderer.frames.values():
                cv2.putText(
                    frames[-1],
                    f"Wind Angle: {round(angle_deg)} deg",
                    (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 0, 0),
                    2,
                )

    sim.set_wind(0, 0)

    for _ in range(1000):
        sim.step()
        sim.render_as_needed()

sim.renderer.show_in_notebook()

nmf/front_cam This browser does not support the video tag.


nmf/top_cam This browser does not support the video tag.


nmf/right_cam This browser does not support the video tag.


We can retrieve the joint state (position in quaternion format, angular velocities and accelerations) and torques with `sim.get_antenna_data`:

In [5]:
sim.get_antenna_data(fly.name)

{'l': {'qpos': array([ 0.99784931, -0.06431949,  0.01159979,  0.00502095]),
  'qvel': array([ 0.45647668,  0.56760959, -0.01446059]),
  'qacc': array([-16.00091514, -22.15418647,   4.73208243]),
  'qfrc_passive': array([ 8.30671971e-04, -7.99964665e-04, -8.61428296e-05])},
 'r': {'qpos': array([ 0.99932577,  0.03106377,  0.01879454, -0.00546067]),
  'qvel': array([0.44984093, 0.37956173, 0.09994724]),
  'qacc': array([-13.28875976, -13.45844387,  -3.48936196]),
  'qfrc_passive': array([-1.07141939e-03, -7.55671397e-04,  9.29898148e-06])}}